
<div style="background:linear-gradient(120deg,#6C5CE7,#0984E3);padding:34px 30px;border-radius:18px;color:white;margin-bottom:18px;">
  <div style="font-size:0.85rem;letter-spacing:2px;text-transform:uppercase;opacity:0.85;">Mineria de Textos - Fase 2</div>
  <h1 style="margin:8px 0 6px 0;font-size:2.2rem;">Analisis semantico con BETO</h1>
  <p style="margin:0;font-size:1.05rem;opacity:0.95;">Embeddings contextuales, busqueda semantica y Masked Language Modeling sobre reseñas de cine y series en español.</p>
</div>

**Objetivo:** entender que aporta BETO frente a representaciones estaticas como Word2Vec, como se interpretan sus embeddings y que alcance real tienen los analisis realizados.

### Contenido

1. Configuracion y modo de ejecucion
2. El corpus de resenas
3. Ficha de BETO
4. Polisemia contextual
5. Comparacion con Word2Vec
6. Busqueda semantica
7. Masked Language Model (MLM)
8. Mapa 2D de embeddings
9. Conclusiones, limitaciones y ejercicios

> **Importante:** este notebook usa BETO **preentrenado**. No se entrena ni se hace fine-tuning. El corpus se usa solo como fuente de texto para inferencia.



## 1. Configuracion y modo de ejecucion

Este notebook funciona en dos modos:

| Modo | `REGENERAR` | Que hace | Tiempo |
|---|---|---|---|
| Artefactos | `False` | Lee los CSV/NPY ya generados en `data/results/beto/` | Segundos |
| Regenerar | `True` | Ejecuta `src/embeding_beto.py` y recalcula todo con BETO | Varios minutos en CPU |

Para el modo artefactos necesitas haber ejecutado antes:

```bash
python src/embeding_beto.py
```

Dependencias del notebook: `plotly`, `scikit-learn`, `pandas`, `numpy`. Si falta `plotly`, descomenta la primera linea de la siguiente celda.


In [ ]:

# %pip install plotly scikit-learn

import html
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import HTML, display
from plotly.subplots import make_subplots
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# Renderer: en Jupyter usa "notebook"; en PyCharm puedes probar "pycharm"
pio.renderers.default = "notebook"

# ---------------------------------------------------------------------------
# Rutas del proyecto
# ---------------------------------------------------------------------------
RAIZ = Path.cwd()
if not (RAIZ / "src").exists():
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ))

DIR_RESULTADOS = RAIZ / "data" / "results" / "beto"
RUTA_CORPUS = RAIZ / "data" / "processed" / "processed_es_fa.csv"

# True = recalcula todos los artefactos con BETO (lento en CPU)
REGENERAR = False

# ---------------------------------------------------------------------------
# Paleta y estilo
# ---------------------------------------------------------------------------
COLOR_POS = "#00B894"
COLOR_NEG = "#E17055"
COLOR_NEU = "#636E72"
COLOR_ACC = "#6C5CE7"
COLOR_ACC2 = "#0984E3"
PALETA = [
    COLOR_ACC, COLOR_ACC2, COLOR_POS, COLOR_NEG, "#FDCB6E",
    "#E84393", "#00CEC9", "#FAB1A0", "#74B9FF", "#A29BFE",
]
PLANTILLA = "plotly_white"

pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 180)

print("Raiz del proyecto:", RAIZ)
print("Resultados BETO:", DIR_RESULTADOS)


In [ ]:

def titulo_seccion(numero, titulo, subtitulo=""):
    display(HTML(f"""
    <div style="display:flex;align-items:center;gap:14px;margin:28px 0 10px 0;
                border-bottom:2px solid #EDF0F4;padding-bottom:10px;">
      <div style="min-width:42px;height:42px;border-radius:12px;background:{COLOR_ACC};
                  color:white;display:flex;align-items:center;justify-content:center;
                  font-weight:700;font-size:18px;">{numero}</div>
      <div>
        <h2 style="margin:0;font-size:1.45rem;color:#2D3436;">{titulo}</h2>
        <p style="margin:3px 0 0 0;color:#636E72;font-size:0.92rem;">{subtitulo}</p>
      </div>
    </div>
    """))


def nota(texto, tipo="info"):
    estilos = {
        "info": ("#EAF2FF", "#0984E3", "i"),
        "success": ("#E8F8F2", "#00B894", "OK"),
        "warning": ("#FFF4E5", "#E17055", "!"),
    }
    fondo, color, icono = estilos.get(tipo, estilos["info"])
    display(HTML(f"""
    <div style="background:{fondo};border-left:4px solid {color};padding:10px 14px;
                border-radius:8px;margin:10px 0;color:#2D3436;">
      <strong style="color:{color};">{icono}</strong> {texto}
    </div>
    """))


def kpi_html(valor, etiqueta, color=COLOR_ACC):
    return f"""
    <div style="background:white;border:1px solid #E9EDF2;border-top:4px solid {color};
                border-radius:12px;padding:14px 18px;min-width:150px;flex:1;
                box-shadow:0 2px 8px rgba(45,52,54,0.06);">
      <div style="font-size:1.6rem;font-weight:750;color:{color};line-height:1.1;">{valor}</div>
      <div style="font-size:0.82rem;color:#636E72;margin-top:4px;">{etiqueta}</div>
    </div>
    """


def mostrar_kpis(items):
    tarjetas = "".join(kpi_html(*item) for item in items)
    display(HTML(f'<div style="display:flex;gap:12px;flex-wrap:wrap;margin:8px 0 16px 0;">{tarjetas}</div>'))


def estilo_figura(fig, titulo="", subtitulo="", height=420):
    titulo_html = f"<b>{titulo}</b>"
    if subtitulo:
        titulo_html += f"<br><span style='font-size:12px;color:#636E72;'>{subtitulo}</span>"
    fig.update_layout(
        template=PLANTILLA,
        height=height,
        title=dict(text=titulo_html, x=0.02, xanchor="left"),
        margin=dict(l=30, r=30, t=80, b=40),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        font=dict(family="Segoe UI, Arial", size=12, color="#2D3436"),
    )
    fig.update_xaxes(showgrid=False)
    fig.update_yaxes(gridcolor="rgba(0,0,0,0.06)")
    return fig


def mostrar_tabla(df, columnas=None, titulo="", height=None, widths=None):
    datos = df[columnas].copy() if columnas else df.copy()
    datos = datos.fillna("")
    kwargs = {}
    if widths:
        kwargs["columnwidth"] = widths
    fig = go.Figure(data=[go.Table(
        header=dict(
            values=[f"<b>{c}</b>" for c in datos.columns],
            fill_color=COLOR_ACC,
            font=dict(color="white", size=12),
            align="left",
            height=32,
        ),
        cells=dict(
            values=[datos[c].tolist() for c in datos.columns],
            fill_color="white",
            font=dict(size=11, color="#2D3436"),
            align="left",
            height=28,
            line_color="#EDF0F4",
        ),
        **kwargs,
    )])
    alto = height or min(420, 70 + 30 * len(datos))
    fig.update_layout(
        template=PLANTILLA,
        height=alto,
        margin=dict(l=10, r=10, t=50, b=10),
        title=dict(text=f"<b>{titulo}</b>" if titulo else "", x=0.01),
    )
    fig.show()


def tarjeta_ejemplo(palabra, positiva, negativa):
    pos = html.escape(str(positiva))
    neg = html.escape(str(negativa))
    display(HTML(f"""
    <div style="border:1px solid #E9EDF2;border-radius:12px;padding:14px 16px;
                margin:10px 0;background:white;box-shadow:0 2px 8px rgba(45,52,54,0.05);">
      <div style="font-weight:750;color:{COLOR_ACC};margin-bottom:8px;
                  text-transform:capitalize;font-size:1.02rem;">{html.escape(palabra)}</div>
      <div style="border-left:4px solid {COLOR_POS};padding:6px 10px;margin:6px 0;
                  background:#E8F8F2;border-radius:6px;">
        <b style="color:{COLOR_POS};">Contexto positivo</b><br>{pos}
      </div>
      <div style="border-left:4px solid {COLOR_NEG};padding:6px 10px;margin:6px 0;
                  background:#FDEEEB;border-radius:6px;">
        <b style="color:{COLOR_NEG};">Contexto negativo</b><br>{neg}
      </div>
    </div>
    """))


In [ ]:

if REGENERAR:
    import src.embeding_beto as beto
    print("Regenerando analisis con BETO. Esto puede tardar varios minutos en CPU...")
    beto.main()
else:
    print("Modo artefactos: se usaran los resultados ya guardados en", DIR_RESULTADOS)



## 2. El corpus de resenas

El corpus contiene resenas de cine y series en espanol. Cada resena tiene un `rating` de 1 a 10, que usaremos como referencia de polaridad en el analisis de polisemia, pero **no como etiqueta de entrenamiento**.


In [ ]:

titulo_seccion("01", "El corpus de resenas", "3.000 resenas de cine y series en espanol, con rating de 1 a 10.")

df_corpus = pd.read_csv(RUTA_CORPUS)
df_corpus["rating"] = pd.to_numeric(df_corpus["rating"], errors="coerce")
if "n_words" not in df_corpus.columns:
    df_corpus["n_words"] = df_corpus["review_text_es"].astype(str).str.split().str.len()
df_corpus["n_words"] = pd.to_numeric(df_corpus["n_words"], errors="coerce")

mostrar_kpis([
    (f"{len(df_corpus):,}".replace(",", "."), "resenas"),
    (f"{df_corpus['movie_id'].nunique():,}".replace(",", "."), "peliculas / series"),
    (f"{df_corpus['rating'].mean():.1f} / 10", "rating promedio"),
    (f"{df_corpus['n_words'].fillna(0).mean():.0f}", "palabras por resena"),
])


In [ ]:

conteo_rating = df_corpus["rating"].value_counts().sort_index()
top_generos = df_corpus["main_genre"].value_counts().head(10).sort_values()

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Distribucion de ratings", "Generos con mas resenas"),
    column_widths=[0.52, 0.48],
    horizontal_spacing=0.14,
)

fig.add_trace(
    go.Bar(
        x=conteo_rating.index.astype(int).astype(str),
        y=conteo_rating.values,
        marker_color=COLOR_ACC,
        text=conteo_rating.values,
        textposition="outside",
        hovertemplate="Rating %{x}<br>Resenas: %{y}<extra></extra>",
        showlegend=False,
    ),
    row=1, col=1,
)

fig.add_trace(
    go.Bar(
        x=top_generos.values,
        y=top_generos.index,
        orientation="h",
        marker_color=COLOR_ACC2,
        text=top_generos.values,
        textposition="outside",
        hovertemplate="%{y}: %{x} resenas<extra></extra>",
        showlegend=False,
    ),
    row=1, col=2,
)

estilo_figura(fig, "Panorama del corpus", "Ratings y generos predominantes", height=470)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)
fig.show()



**Interpretacion:** el corpus esta concentrado en valoraciones altas (7-10), con una cola de resenas negativas (1-4). Esa distribucion es util para comparar contextos positivos y negativos, pero conviene recordar que las clases no estan balanceadas.

Tambien conviene notar que las resenas son largas (mediana cercana a 159 palabras). Por eso BETO trunca a 256 tokens en los analisis: es una limitacion practica del script.



## 3. Ficha de BETO

**BETO** (`dccuchile/bert-base-spanish-wwm-cased`) es un BERT entrenado desde cero con texto en espanol. Es un modelo **contextual**: la representacion de una palabra depende de la oracion completa, a diferencia de Word2Vec, que asigna un unico vector por palabra.

| Caracteristica | Valor |
|---|---|
| Parametros | 109.260.288 |
| Dimension del embedding | 768 |
| Capas Transformer | 12 |
| Cabezas de atencion | 12 |
| Tipo | BERT base, cased, whole-word masking |
| Uso en este notebook | Inferencia, sin fine-tuning |


In [ ]:

titulo_seccion("02", "Ficha de BETO", "El modelo preentrenado que produce los embeddings y las predicciones.")

mostrar_kpis([
    ("109 M", "parametros"),
    ("768", "dimensiones por token"),
    ("12", "capas Transformer"),
    ("12", "cabezas de atencion"),
])

nota(
    "BETO se usa <b>preentrenado y congelado</b>: no se actualizan sus pesos. "
    "Los analisis son de <b>inferencia</b>, por lo que el corpus no se usa para entrenar.",
    "info",
)



## 4. Polisemia contextual

Una misma palabra puede cambiar de significado segun la oracion. BETO genera un vector distinto para cada aparicion, y aqui comparamos esos vectores con **similitud coseno**.

El script agrupa oraciones con `rating >= 7` como contexto positivo y `rating <= 4` como negativo. Despues calcula:

- similitud promedio dentro de las positivas,
- similitud promedio dentro de las negativas,
- similitud promedio entre ambos grupos,
- y el ejemplo mas representativo de cada grupo segun la cercania a su centroide.

Si la similitud **entre polaridades** es baja, la palabra esta cambiando de contexto mas claramente.


In [ ]:

titulo_seccion("03", "Polisemia contextual", "La misma palabra, distintos vectores segun la oracion.")

df_poli = pd.read_csv(DIR_RESULTADOS / "polisemia_resumen.csv")
df_poli_tabla = df_poli.copy()
for columna in [
    "sim_promedio_positivas", "sim_promedio_negativas",
    "sim_promedio_entre_polaridades", "sim_promedio_global",
    "sim_min_global", "sim_max_global",
]:
    df_poli_tabla[columna] = df_poli_tabla[columna].map(
        lambda v: f"{v:.4f}" if pd.notna(v) else "-"
    )

mostrar_tabla(
    df_poli_tabla,
    ["palabra", "n_positivas", "n_negativas",
     "sim_promedio_positivas", "sim_promedio_negativas",
     "sim_promedio_entre_polaridades"],
    titulo="Similitud coseno promedio por palabra",
    height=310,
    widths=[0.16, 0.12, 0.12, 0.20, 0.20, 0.20],
)


In [ ]:

fig = go.Figure()
fig.add_trace(go.Bar(
    name="Positivas",
    x=df_poli["palabra"],
    y=df_poli["sim_promedio_positivas"],
    marker_color=COLOR_POS,
    text=df_poli["sim_promedio_positivas"].map(lambda v: f"{v:.3f}"),
    textposition="outside",
))
fig.add_trace(go.Bar(
    name="Negativas",
    x=df_poli["palabra"],
    y=df_poli["sim_promedio_negativas"],
    marker_color=COLOR_NEG,
    text=df_poli["sim_promedio_negativas"].map(lambda v: f"{v:.3f}"),
    textposition="outside",
))
fig.add_trace(go.Bar(
    name="Entre polaridades",
    x=df_poli["palabra"],
    y=df_poli["sim_promedio_entre_polaridades"],
    marker_color=COLOR_NEU,
    text=df_poli["sim_promedio_entre_polaridades"].map(lambda v: f"{v:.3f}"),
    textposition="outside",
))

valores = pd.concat([
    df_poli["sim_promedio_positivas"],
    df_poli["sim_promedio_negativas"],
    df_poli["sim_promedio_entre_polaridades"],
])
estilo_figura(
    fig,
    "Similitud contextual por palabra",
    "Barras mas bajas entre polaridades indican mayor cambio de contexto",
    height=470,
)
fig.update_layout(
    barmode="group",
    yaxis=dict(
        title="Similitud coseno",
        range=[max(0.0, float(valores.min()) - 0.05), min(1.0, float(valores.max()) + 0.05)],
    ),
    xaxis=dict(title=""),
)
fig.show()


In [ ]:

for _, fila in df_poli.iterrows():
    tarjeta_ejemplo(fila["palabra"], fila["oracion_positiva"], fila["oracion_negativa"])



**Interpretacion:** las similitudes dentro de cada polaridad suelen ser mas altas que la similitud entre polaridades. Eso indica que BETO representa la palabra de forma distinta cuando aparece en contextos positivos y negativos. La diferencia no es enorme porque la palabra conserva parte de su significado central, pero el cambio es medible.

**Limitacion:** la polaridad de cada grupo se apoya en el `rating` de la resena, no en un clasificador de sentimiento. BETO aporta el embedding contextual y la cercania a los centroides, pero no etiqueta la resena.



## 5. Comparacion con Word2Vec

Word2Vec asigna **un solo vector por palabra**, sin importar la oracion. BETO, en cambio, produce un vector distinto para cada aparicion. Esta seccion compara esa diferencia conceptual con los resultados disponibles.

Si el archivo `data/models/word2vec_resenas.kv` no existe, el script omite la comparacion y no se genera `comparacion_word2vec.csv`.


In [ ]:

titulo_seccion("04", "BETO vs Word2Vec", "Contextual frente a estatico: que cambia y que no.")

ruta_w2v = DIR_RESULTADOS / "comparacion_word2vec.csv"

if ruta_w2v.exists():
    df_w2v = pd.read_csv(ruta_w2v)
    mostrar_tabla(
        df_w2v,
        ["palabra", "sim_promedio_beto", "sim_promedio_word2vec", "en_vocabulario_word2vec"],
        titulo="Comparacion disponible",
        height=300,
    )
else:
    df_w2v = pd.DataFrame()
    nota(
        "No se encontro <code>comparacion_word2vec.csv</code> porque falta "
        "<code>data/models/word2vec_resenas.kv</code>. La comparacion se omite, "
        "pero el resto del analisis no se ve afectado.",
        "warning",
    )



| Aspecto | Word2Vec | BETO |
|---|---|---|
| Unidad | Vector unico por palabra | Vector por aparicion |
| Contexto | No cambia con la oracion | Cambia con la oracion |
| Similitud de una palabra consigo misma | Siempre 1.0 | Puede ser menor a 1.0 |
| Uso tipico | Similitud lexica, analogias | Analisis contextual, busqueda semantica |

La comparacion no busca decir que un modelo es "mejor", sino mostrar que tipo de informacion captura cada uno.



## 6. Busqueda semantica

El buscador convierte cada resena en un vector promediando los embeddings de sus tokens (**mean pooling**) y normaliza el resultado. Luego compara la consulta con cada resena usando **similitud coseno**.

La muestra contiene 300 resenas. Para cada una de las tres consultas de ejemplo se muestran las 5 resenas mas parecidas.


In [ ]:

titulo_seccion("05", "Busqueda semantica", "Consulta en lenguaje natural y resenas mas parecidas.")

ruta_busqueda = DIR_RESULTADOS / "busqueda_resenas.csv"

if ruta_busqueda.exists():
    df_busqueda = pd.read_csv(ruta_busqueda)
    print(f"Consultas: {df_busqueda['consulta'].nunique()} | resultados: {len(df_busqueda)}")
else:
    df_busqueda = pd.DataFrame()
    nota(
        "Falta <code>busqueda_resenas.csv</code>. Ejecuta "
        "<code>python src/embeding_beto.py</code> para generarlo.",
        "warning",
    )


In [ ]:

if not df_busqueda.empty:
    consultas = list(dict.fromkeys(df_busqueda["consulta"]))
    fig = go.Figure()
    visibilidad = []
    botones = []
    todos_los_valores = []

    for i, consulta in enumerate(consultas):
        sub = df_busqueda[df_busqueda["consulta"] == consulta].head(5).reset_index(drop=True)
        sub["rank"] = np.arange(1, len(sub) + 1)
        sub = sub.iloc[::-1]

        etiquetas = [
            f"#{int(r)} | rating {rt} | {g}"
            for r, rt, g in zip(sub["rank"], sub["rating"], sub["genero"])
        ]

        fig.add_trace(go.Bar(
            x=sub["similitud"],
            y=etiquetas,
            orientation="h",
            marker=dict(
                color=sub["similitud"],
                colorscale=[[0, "#DFE6E9"], [1, COLOR_ACC]],
                showscale=False,
            ),
            text=[f"{v:.3f}" for v in sub["similitud"]],
            textposition="outside",
            customdata=sub[["fragmento"]].values,
            hovertemplate="<b>%{y}</b><br>Similitud: %{x:.4f}<br>%{customdata[0]}<extra></extra>",
            name=consulta,
            visible=(i == 0),
        ))
        todos_los_valores.extend(sub["similitud"].tolist())
        visibilidad.append([j == i for j in range(len(consultas))])

    for i, consulta in enumerate(consultas):
        botones.append(dict(
            label=consulta[:38] + ("..." if len(consulta) > 38 else ""),
            method="update",
            args=[
                {"visible": visibilidad[i]},
                {"title": (
                    "<b>Busqueda semantica</b><br>"
                    f"<span style='font-size:12px;color:#636E72;'>Consulta: \"{consulta}\"</span>"
                )},
            ],
        ))

    bajo = max(0.0, min(todos_los_valores) - 0.02)
    alto = min(1.0, max(todos_los_valores) + 0.02)

    fig.update_layout(
        template=PLANTILLA,
        height=450,
        title=(
            "<b>Busqueda semantica</b><br>"
            f"<span style='font-size:12px;color:#636E72;'>Consulta: \"{consultas[0]}\"</span>"
        ),
        margin=dict(l=20, r=60, t=100, b=20),
        updatemenus=[dict(
            active=0,
            buttons=botones,
            x=1,
            y=1.18,
            xanchor="right",
            showactive=True,
        )],
        xaxis=dict(
            title="Similitud coseno",
            range=[bajo, alto],
            showgrid=True,
            gridcolor="rgba(0,0,0,0.06)",
        ),
        yaxis=dict(title="", showgrid=False),
        showlegend=False,
        font=dict(family="Segoe UI, Arial", size=12, color="#2D3436"),
    )
    fig.show()


In [ ]:

if not df_busqueda.empty:
    tabla_busqueda = df_busqueda.copy()
    tabla_busqueda["fragmento"] = (
        tabla_busqueda["fragmento"].astype(str).str.slice(0, 110) + "..."
    )
    tabla_busqueda["similitud"] = tabla_busqueda["similitud"].map(lambda v: f"{v:.4f}")
    mostrar_tabla(
        tabla_busqueda,
        ["consulta", "similitud", "rating", "genero", "fragmento"],
        titulo="Top-5 por consulta",
        height=620,
        widths=[0.22, 0.08, 0.07, 0.09, 0.54],
    )



**Interpretacion:** las similitudes son altas (cerca de 0.86-0.90) porque todas las resenas comparten el dominio del cine. Aun asi, el orden cambia segun la consulta: BETO captura temas y tono, no solo palabras exactas.

**Limitacion:** el mean pooling aplana la informacion de la resena completa. Para textos largos, una sola parte del texto puede dominar el vector.



## 7. Masked Language Model (MLM)

BETO fue preentrenado, entre otras tareas, para predecir palabras ocultas. Aqui se enmascara una posicion con `[MASK]` en plantillas tipicas de resenas y se muestran las 5 predicciones mas probables.

La columna `sentimiento_estimado` es una **heuristica** basada en raices de palabras (`RAICES_POSITIVAS` y `RAICES_NEGATIVAS` del script). No es una salida de BETO ni un clasificador de sentimiento.


In [ ]:

titulo_seccion("06", "Masked Language Model", "Que palabras completa BETO en frases tipicas de resenas.")

df_mlm = pd.read_csv(DIR_RESULTADOS / "mlm_predicciones.csv")
print(f"Plantillas: {df_mlm['plantilla'].nunique()} | predicciones: {len(df_mlm)}")


In [ ]:

if not df_mlm.empty:
    plantillas = list(dict.fromkeys(df_mlm["plantilla"]))
    mapa_color = {
        "positiva": COLOR_POS,
        "negativa": COLOR_NEG,
        "neutra/otra": COLOR_NEU,
    }

    fig = go.Figure()
    visibilidad = []
    botones = []

    for i, plantilla in enumerate(plantillas):
        sub = df_mlm[df_mlm["plantilla"] == plantilla].sort_values("ranking").head(5)
        colores = [mapa_color.get(s, COLOR_NEU) for s in sub["sentimiento_estimado"]]
        sub_r = sub.iloc[::-1]
        colores_r = colores[::-1]

        fig.add_trace(go.Bar(
            x=sub_r["probabilidad"],
            y=[f"{int(r)}. {p}" for r, p in zip(sub_r["ranking"], sub_r["prediccion"])],
            orientation="h",
            marker_color=colores_r,
            text=[f"{v:.1%}" for v in sub_r["probabilidad"]],
            textposition="outside",
            hovertemplate="<b>%{y}</b><br>Probabilidad: %{x:.2%}<extra></extra>",
            name=plantilla[:30],
            visible=(i == 0),
        ))
        visibilidad.append([j == i for j in range(len(plantillas))])

    for i, plantilla in enumerate(plantillas):
        botones.append(dict(
            label=plantilla[:45] + ("..." if len(plantilla) > 45 else ""),
            method="update",
            args=[
                {"visible": visibilidad[i]},
                {"title": (
                    "<b>Predicciones MLM</b><br>"
                    f"<span style='font-size:12px;color:#636E72;'>{plantilla}</span>"
                )},
            ],
        ))

    fig.update_layout(
        template=PLANTILLA,
        height=470,
        title=(
            "<b>Predicciones MLM</b><br>"
            f"<span style='font-size:12px;color:#636E72;'>{plantillas[0]}</span>"
        ),
        margin=dict(l=20, r=60, t=100, b=20),
        updatemenus=[dict(
            active=0,
            buttons=botones,
            x=1,
            y=1.18,
            xanchor="right",
            showactive=True,
        )],
        xaxis=dict(title="Probabilidad", tickformat=".0%", showgrid=True,
                   gridcolor="rgba(0,0,0,0.06)"),
        yaxis=dict(title="", showgrid=False),
        showlegend=False,
        font=dict(family="Segoe UI, Arial", size=12, color="#2D3436"),
    )
    fig.show()


In [ ]:

if not df_mlm.empty:
    primeras = list(dict.fromkeys(df_mlm["plantilla"]))[:3]
    tabla_mlm = df_mlm[df_mlm["plantilla"].isin(primeras)].copy()
    tabla_mlm["plantilla"] = tabla_mlm["plantilla"].map(
        lambda s: s[:58] + "..." if len(s) > 58 else s
    )
    tabla_mlm["probabilidad"] = tabla_mlm["probabilidad"].map(lambda v: f"{v:.2%}")
    mostrar_tabla(
        tabla_mlm,
        ["plantilla", "ranking", "prediccion", "probabilidad", "sentimiento_estimado"],
        titulo="Primeras plantillas y sus predicciones",
        height=560,
        widths=[0.42, 0.08, 0.18, 0.12, 0.20],
    )

    nota(
        "Las etiquetas <b>positiva</b>/<b>negativa</b> se asignan por coincidencia de prefijos "
        "con listas de raices. Son una ayuda interpretativa, no una clasificacion real de sentimiento.",
        "warning",
    )



**Interpretacion:** BETO completa las plantillas con palabras coherentes con el contexto. Por ejemplo, en "El ritmo ... es demasiado [MASK]" predice *lento* con alta probabilidad, y en "Salio del cine con una sensacion [MASK], no la recomiendo" aparecen *terrible* y *horrible*.

**Limitacion:** el modelo no "entiende" sentimiento de forma supervisada; solo modela probabilidades del lenguaje. Las plantillas estan fijas y la evaluacion de sentimiento es heuristica.



## 8. Mapa 2D de embeddings

Aqui proyectamos los 300 embeddings de resenas a dos dimensiones con **PCA** y **t-SNE**. Cada punto es una resena. El color representa el rating o el genero.

El objetivo es visual: ver si resenas parecidas quedan cerca. No es una prueba de que BETO "sepa" el rating; solo muestra la estructura de los embeddings generados.


In [ ]:

titulo_seccion("07", "Mapa 2D de embeddings", "Como se distribuyen las resenas en el espacio semantico de BETO.")

embeddings = np.load(DIR_RESULTADOS / "embeddings_resenas.npy")
indice = pd.read_csv(DIR_RESULTADOS / "resenas_indice.csv")
indice["rating"] = pd.to_numeric(indice["rating"], errors="coerce")

assert len(embeddings) == len(indice), "Embeddings e indice no coinciden"

pca = PCA(n_components=2, random_state=42)
coords_pca = pca.fit_transform(embeddings)

tsne = TSNE(
    n_components=2,
    perplexity=min(30, len(embeddings) - 1),
    init="pca",
    learning_rate="auto",
    random_state=42,
)
coords_tsne = tsne.fit_transform(embeddings)

print(f"Embeddings: {embeddings.shape[0]} resenas x {embeddings.shape[1]} dimensiones")
print(f"Varianza explicada por PCA (2 componentes): {pca.explained_variance_ratio_.sum():.1%}")


In [ ]:

hover = np.stack([
    indice["rating"].astype(str),
    indice["main_genre"].fillna("-").astype(str),
    indice["review_summary"].fillna("").astype(str).str.slice(0, 90),
], axis=-1)

fig = make_subplots(rows=1, cols=2, subplot_titles=("PCA", "t-SNE"))

for coords, columna in [(coords_pca, 1), (coords_tsne, 2)]:
    fig.add_trace(
        go.Scatter(
            x=coords[:, 0],
            y=coords[:, 1],
            mode="markers",
            marker=dict(
                size=8,
                color=indice["rating"],
                colorscale="RdYlGn",
                cmin=1,
                cmax=10,
                colorbar=dict(title="Rating", len=0.75) if columna == 1 else None,
                line=dict(width=0.4, color="white"),
                opacity=0.85,
            ),
            customdata=hover,
            hovertemplate=(
                "<b>Rating %{customdata[0]}</b> | %{customdata[1]}"
                "<br>%{customdata[2]}<extra></extra>"
            ),
            showlegend=False,
        ),
        row=1,
        col=columna,
    )

estilo_figura(
    fig,
    "Mapa 2D de embeddings de resenas",
    "Cada punto es una resena; el color indica su rating",
    height=490,
)
fig.update_xaxes(title="", showgrid=False)
fig.update_yaxes(title="", showgrid=False)
fig.show()


In [ ]:

df_mapa = pd.DataFrame({
    "x": coords_tsne[:, 0],
    "y": coords_tsne[:, 1],
    "rating": indice["rating"],
    "genero": indice["main_genre"].fillna("-"),
    "resumen": indice["review_summary"].fillna("").str.slice(0, 90),
})

fig = px.scatter(
    df_mapa,
    x="x",
    y="y",
    color="genero",
    hover_data={"rating": True, "resumen": True, "x": False, "y": False},
    color_discrete_sequence=PALETA,
)
estilo_figura(
    fig,
    "t-SNE coloreado por genero",
    "Los grupos no son perfectos: BETO captura semantica, no sentimiento",
    height=490,
)
fig.update_traces(
    marker=dict(size=8, line=dict(width=0.4, color="white"), opacity=0.85)
)
fig.update_xaxes(title="", showgrid=False)
fig.update_yaxes(title="", showgrid=False)
fig.show()



**Interpretacion:** PCA resume la mayor variabilidad en dos ejes, mientras que t-SNE prioriza vecindades locales. Si algunas resenas de rating similar se agrupan, eso sugiere que el embedding captura parte del tono; si se mezclan, es esperable, porque BETO no fue entrenado para predecir ratings.

**Limitacion:** PCA y t-SNE son proyecciones aproximadas. Que dos puntos esten cerca en 2D no garantiza que esten cerca en las 768 dimensiones originales.



## 9. Conclusiones, limitaciones y ejercicios

### Que demuestra el analisis

- BETO produce **embeddings contextuales**: la misma palabra cambia de vector segun la oracion.
- La busqueda semantica funciona sin coincidencia exacta de palabras, usando similitud coseno.
- El MLM muestra el conocimiento linguistico del modelo preentrenado.
- PCA y t-SNE permiten explorar visualmente la estructura del espacio de embeddings.

### Limitaciones

- BETO **no fue entrenado** con este corpus y no se hizo fine-tuning.
- El rating se usa como referencia de polaridad, no como etiqueta supervisada.
- BETO **no es un clasificador de sentimiento**; las raices positivas/negativas son una heuristica.
- La busqueda usa una muestra de 300 resenas y mean pooling, que aplana textos largos.
- La comparacion con Word2Vec depende de que exista el modelo `word2vec_resenas.kv`.

### Ejercicios propuestos

1. Cambia `N_RESENAS_BUSQUEDA` en el script a `None` y compara los resultados del buscador con el corpus completo.
2. Agrega una nueva palabra a `PALABRAS_POLISEMIA` y observa si la similitud entre polaridades baja.
3. Sustituye las plantillas MLM por frases de tu propio interes y analiza las predicciones.
4. Reemplaza la heuristica de raices por un modelo de sentimiento en espanol (por ejemplo, `pysentimiento/robertuito-sentiment-analysis`) y compara las etiquetas.
5. Fine-tunea un clasificador sobre BETO usando el rating como etiqueta y compara con el uso solo de inferencia.
